# Triton Kernel 主线 · 第 8/10 课：二维 Tile 与转置

> 状态：**学习中（待提交）**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：构造二维 pointer tensor，正确处理非方阵与双向 mask。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：Python、PyTorch 张量、CUDA 基本线程/内存概念
- 本课在路线中的作用：二维 tile 用广播后的行/列 offset 构造 pointer matrix；`tl.trans` 交换编译期维度。

## 核心心智模型

### 1. 它是什么，解决什么问题

二维 tile 用广播后的行/列 offset 构造 pointer matrix；`tl.trans` 交换编译期维度。

### 2. 它如何工作

输入地址是 rm[:,None]*sxm+rn[None,:]*sxn，输出地址交换 rm/rn 并使用输出 strides。

### 3. 正确性条件与常见误区

输入/输出 mask shape 必须与各自 tile shape一致；只测方阵会掩盖 stride 错。

### 4. 性能与工程取舍

32×32 tile 通用但未必最佳；布局、dtype 与 cache 行为影响选择。

## 具体演示

31×65 会产生 1×3 个 tiles，边界 tile 需要在 M/N 两维 mask。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐转置值。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def transpose_kernel(x, out, M: tl.constexpr, N: tl.constexpr,
                     sxm: tl.constexpr, sxn: tl.constexpr,
                     som: tl.constexpr, son: tl.constexpr,
                     BM: tl.constexpr, BN: tl.constexpr):
    rm = tl.program_id(0) * BM + tl.arange(0, BM)
    rn = tl.program_id(1) * BN + tl.arange(0, BN)
    ptrs = x + rm[:, None] * sxm + rn[None, :] * sxn
    mask = (rm[:, None] < M) & (rn[None, :] < N)
    tile = tl.load(ptrs, mask=mask)
    out_ptrs = out + rn[:, None] * som + rm[None, :] * son
    out_mask = (rn[:, None] < N) & (rm[None, :] < M)
    tl.store(out_ptrs, ______, mask=out_mask)  # TODO: 交换 tile 两维

def transpose(x):
    assert x.ndim == 2
    M, N = x.shape
    out = torch.empty((N, M), device=x.device, dtype=x.dtype)
    grid = (triton.cdiv(M, 32), triton.cdiv(N, 32))
    transpose_kernel[grid](x, out, M, N, x.stride(0), x.stride(1),
                           out.stride(0), out.stride(1), BM=32, BN=32)
    return out

for shape in ((2, 3), (31, 65), (128, 96)):
    x = torch.randn(shape, device="cuda")
    torch.testing.assert_close(transpose(x), x.T)


### 检查方法

在 CUDA/Triton 环境运行本单元格；断言覆盖规则尺寸和非规则尾块。首次 JIT 不计入性能。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“二维 Tile 与转置”的工作机制。

**你的答案：**


### Q2

输出 mask 为何不能直接复用输入 mask？

**你的答案：**


### Q3

对非连续输入，这个 stride-aware kernel 还需哪些 wrapper 检查？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考资料

- [Triton Tutorials](https://triton-lang.org/main/getting-started/tutorials/)
- [Triton language API](https://triton-lang.org/main/python-api/triton.language.html)

资料用于建立事实基线；面试回答仍需用自己的语言组织。